<a href="https://colab.research.google.com/github/mehrerm/TFM/blob/main/notebooks/exploracion_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



En esta aplicación implementaremos un modelo de puntuación de riesgo poblacional a la mortalidad debido a los cánceres más comunes y su relación con las cantidades y tecnologías usadas en radioterapia por país.

In [110]:
#Cargo o importo pandas, numpy, Matplotlib,
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import statsmodels.api as sm




import requests
import unicodedata
import os
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

!pip install optbinning
from optbinning import OptimalBinning, Scorecard, BinningProcess
from sklearn.linear_model import LogisticRegression
from urllib.parse import quote




#lugar donde se van guardando las figuras

output_path = Path("TFM/data/processed")
output_path.mkdir(parents=True, exist_ok=True)

fig_dir = "figuras"
os.makedirs(fig_dir, exist_ok=True)

#########################

Se inicia cargando los datasets a explorar, se tiene tanto los datos de todos los equipos, tipos de cancar y años, como el que se pretende estudiar que sería el del año 2022, se selecciona este porque es el más reciente donde se puede constatar que los equipos están actualizados así como una cantidad suficientes de eventos para el estudio.

In [111]:
df_merged = pd.read_csv("data/processed/df_merged.csv")
df_merged_2022 = pd.read_csv("data/processed/df_merged2022.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/df_merged2022.csv'

In [ ]:
dups_2022 = (
    df_all_cancer[df_all_cancer["Year"] == 2022]
    .loc[lambda x: x.duplicated(subset=key_cols, keep=False)]
    .sort_values(key_cols)
)

dups_2022.head(20)

Los datos se encuentran limpios y fucionados en un solo dataframe. Ahora se realiza una exploración previa.

In [ ]:
#Tras una exploración previa, se determinó que lo ideal es tomar la muestra post
#Es un solo año el que se recomienda utilizar, ya que el Last update solamente
#muestra un solo año, sin información de años anteriores sobre equipamientos

#top 5 países con más equipos
columnas_equipos = ["RTCenters", "Linac", "Protontherapy", "XRay", "Brachytherapy"]
anios = [2022]

# Copia de trabajo
df_tmp = df_merged.copy()

# Asegurar columnas numéricas
for c in columnas_equipos + ["Population"]:
    if c in df_tmp.columns:
        df_tmp[c] = pd.to_numeric(df_tmp[c], errors="coerce")

for year in anios:
    print("\n" + "=" * 70)
    print(f"TOP 5 países por equipo — Año {year}")
    print("=" * 70)

    # 1) Filtrar por año
    df_year = df_tmp[df_tmp["Year"] == year].copy()

    # 2) Colapsar a una fila por país-año
    #    max() evita inflar valores por duplicados del merge
    equipos_pais = (
        df_year
        .groupby("Country_harmonized", as_index=False)
        .agg(
            {**{c: "max" for c in columnas_equipos},
             "Population": "max",
             "Last Update": "max"}
        )
    )

    # 3) Top 5 por cada equipo
    for col in columnas_equipos:
        print(f"\nTop 5 países por: {col} ({year})")
        top5 = (
            equipos_pais[["Country_harmonized", "Population", "Last Update", col]]
            .dropna(subset=[col])
            .sort_values(by=col, ascending=False)
            .head(5)
        )
        print(top5.to_string(index=False))


Antes de abordar los análisis de relación, se realizó un análisis exploratorio visual estratificado por año y sexo, con el objetivo de contextualizar la evolución temporal de la incidencia y la mortalidad por cáncer. Este análisis permitió identificar el periodo más adecuado para el estudio principal, teniendo en cuenta posibles distorsiones en los indicadores de salud asociadas a acontecimientos excepcionales, como la pandemia de COVID-19 en el 2020, que alteró significativamente los patrones de diagnóstico y tratamiento. Con esto además, podemos visualizar como evoluciona la incidencia y mortalidad en el timepo, si existen diferencias sistemáticas por sexo y detectar patrones globales.



In [ ]:
df_trend = (
    df_merged
    .dropna(subset=["Year", "ASR (World)", "Sex", "Type"])
    .groupby(["Year", "Sex", "Type"], as_index=False)
    .agg({"ASR (World)": "mean"})
    .sort_values("Year")
)

%matplotlib inline

plt.figure(figsize=(10,5))

for sex in ["Male", "Female"]:
    for t in ["Incidence", "Mortality"]:
        sub = df_trend[(df_trend["Sex"] == sex) & (df_trend["Type"] == t)]
        plt.plot(
            sub["Year"],
            sub["ASR (World)"],
            marker="o",
            label=f"{sex} – {t}"
        )

plt.xlabel("Year")
plt.ylabel("ASR (World)")
plt.title("Evolución temporal de la ASR (World) por sexo e indicador")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig01_figure_ASR_anno.jpg", dpi=300, bbox_inches="tight")
plt.show()


#Se grafica entonces la evolución temporal de la ASR, estratificado por sexo

In [ ]:

df_trend = (
    df_merged
    .dropna(subset=["Year", "ASR (World)", "Sex", "Type"])
    .groupby(["Year", "Sex", "Type"], as_index=False)
    .agg({"ASR (World)": "mean"})
    .sort_values("Year")
)

%matplotlib inline

plt.figure(figsize=(10, 5))

for sex in ["Male", "Female"]:
    for t in ["Incidence", "Mortality"]:

        sub = df_trend[
            (df_trend["Sex"] == sex) &
            (df_trend["Type"] == t)
        ].sort_values("Year")

        if sub.empty or len(sub) < 2:
            continue

        # --- Serie observada ---
        plt.plot(
            sub["Year"],
            sub["ASR (World)"],
            marker="o",
            label=f"{sex} – {t}"
        )

        # --- Recta de regresión lineal ---
        b, a = np.polyfit(sub["Year"], sub["ASR (World)"], 1)

        plt.plot(
            sub["Year"],
            a + b * sub["Year"],
            linestyle="--",
            alpha=0.7
        )

plt.xlabel("Year")
plt.ylabel("ASR (World)")
plt.title("Evolución temporal de la ASR (World) por sexo e indicador")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig01_figure_ASR_anno.jpg", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
df_trend = (
    df_merged
    .dropna(subset=["Year", "ASR (World)", "Sex", "Type"])
    .groupby(["Year", "Sex", "Type"], as_index=False)
    .agg({"ASR (World)": "mean"})
    .sort_values("Year")
)

%matplotlib inline

plt.figure(figsize=(10, 5))

for sex in ["Male", "Female"]:
    for t in ["Incidence", "Mortality"]:

        sub = df_trend[
            (df_trend["Sex"] == sex) &
            (df_trend["Type"] == t)
        ].sort_values("Year")

        if len(sub) < 2:
            continue

        # --- Regresión lineal ---
        b, a = np.polyfit(sub["Year"], sub["ASR (World)"], 1)

        # --- Serie observada ---
        plt.plot(
            sub["Year"],
            sub["ASR (World)"],
            marker="o",
            label=f"{sex} – {t} (β = {b:.2f})"
        )

        # --- Recta de tendencia ---
        plt.plot(
            sub["Year"],
            a + b * sub["Year"],
            linestyle="--",
            alpha=0.7
        )

plt.xlabel("Year")
plt.ylabel("ASR (World)")
plt.title("Evolución temporal de la ASR (World) por sexo e indicador")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"{fig_dir}/fig01_figure_ASR_anno.jpg", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 1) Preparar tendencia: ahora agrupamos también por Cancer label
df_trend_cancer = (
    df_merged
    .dropna(subset=["Year", "ASR (World)", "Sex", "Type", "Cancer label"])
    .groupby(["Cancer label", "Year", "Sex", "Type"], as_index=False)
    .agg({"ASR (World)": "mean"})
    .sort_values(["Cancer label", "Year"])
)

# (opcional) orden de cánceres para que salga consistente
orden_cancer = ["Lung", "Breast", "Prostate", "Cervix", "Lip_oc_Pharynx"]
#orden_cancer = ["Lung", "Breast", "Prostate", "Colon", "Cervix", "Leukaemia"]
cancers = [c for c in orden_cancer if c in df_trend_cancer["Cancer label"].unique()]
# si hubiera otros, los añadimos al final
cancers += [c for c in df_trend_cancer["Cancer label"].unique() if c not in cancers]

# 2) Graficar una figura por cáncer
os.makedirs(fig_dir, exist_ok=True)

for cancer in cancers:
    df_c = df_trend_cancer[df_trend_cancer["Cancer label"] == cancer]

    plt.figure(figsize=(10, 5))

    for sex in ["Male", "Female"]:
        for t in ["Incidence", "Mortality"]:
            sub = df_c[(df_c["Sex"] == sex) & (df_c["Type"] == t)].sort_values("Year")

            # Si no hay datos para esa combinación, saltamos
            if sub.empty:
                continue

            plt.plot(
                sub["Year"],
                sub["ASR (World)"],
                marker="o",
                label=f"{sex} – {t}"
            )

    plt.xlabel("Year")
    plt.ylabel("ASR (World)")
    plt.title(f"Evolución temporal de la ASR (World) — {cancer} (por sexo e indicador)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # nombre de archivo seguro
    cancer_safe = str(cancer).replace(" ", "_").replace("/", "_")
    plt.savefig(f"{fig_dir}/fig_ASR_anno_{cancer_safe}.jpg", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# 1) Preparar tendencia: ahora agrupamos también por Cancer label
df_trend_cancer = (
    df_merged
    .dropna(subset=["Year", "ASR (World)", "Sex", "Type", "Cancer label"])
    .groupby(["Cancer label", "Year", "Sex", "Type"], as_index=False)
    .agg({"ASR (World)": "mean"})
    .sort_values(["Cancer label", "Year"])
)

# (opcional) orden de cánceres para que salga consistente
#orden_cancer = ["Lung", "Breast", "Prostate", "Colon", "Cervix", "Leukaemia"]
orden_cancer = ["Lung", "Breast", "Prostate", "Cervix", "Lip_oc_Pharynx"]
cancers = [c for c in orden_cancer if c in df_trend_cancer["Cancer label"].unique()]
cancers += [c for c in df_trend_cancer["Cancer label"].unique() if c not in cancers]

# 2) Graficar una figura por cáncer
os.makedirs(fig_dir, exist_ok=True)

for cancer in cancers:
    df_c = df_trend_cancer[df_trend_cancer["Cancer label"] == cancer]

    plt.figure(figsize=(10, 5))

    for sex in ["Male", "Female"]:
        for t in ["Incidence", "Mortality"]:

            sub = df_c[
                (df_c["Sex"] == sex) &
                (df_c["Type"] == t)
            ].sort_values("Year")

            # Si no hay datos o hay menos de 2 puntos, saltamos (no se puede ajustar recta)
            if len(sub) < 2:
                continue

            # --- Ajuste lineal: ASR = a + b * Year ---
            b, a = np.polyfit(sub["Year"], sub["ASR (World)"], 1)

            # --- Serie observada + pendiente en leyenda ---
            plt.plot(
                sub["Year"],
                sub["ASR (World)"],
                marker="o",
                label=f"{sex} – {t} (β={b:.2f})"
            )

            # --- Recta de tendencia (discontinua) ---
            plt.plot(
                sub["Year"],
                a + b * sub["Year"],
                linestyle="--",
                alpha=0.7
            )

    plt.xlabel("Year")
    plt.ylabel("ASR (World)")
    plt.title(f"Evolución temporal de la ASR (World) — {cancer} (por sexo e indicador)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # nombre de archivo seguro
    cancer_safe = str(cancer).replace(" ", "_").replace("/", "_")
    plt.savefig(f"{fig_dir}/fig_ASR_anno_{cancer_safe}.jpg", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
slopes = []

for cancer in cancers:
    for sex in ["Male", "Female"]:
        for t in ["Incidence", "Mortality"]:

            sub = df_trend_cancer[
                (df_trend_cancer["Cancer label"] == cancer) &
                (df_trend_cancer["Sex"] == sex) &
                (df_trend_cancer["Type"] == t)
            ].sort_values("Year")

            if len(sub) < 2:
                continue

            # Ajuste lineal: ASR = a + b * Year
            b, a = np.polyfit(sub["Year"], sub["ASR (World)"], 1)

            slopes.append({
                "Cancer label": cancer,
                "Sex": sex,
                "Type": t,
                "Slope_ASR_per_year": b
            })

df_slopes = pd.DataFrame(slopes)



df_slopes_incidence = df_slopes[df_slopes["Type"] == "Incidence"]
df_slopes_mortality = df_slopes[df_slopes["Type"] == "Mortality"]



df_slopes_incidence = df_slopes_incidence.sort_values(
    by="Slope_ASR_per_year",
    ascending=False
)

df_slopes_mortality = df_slopes_mortality.sort_values(
    by="Slope_ASR_per_year",
    ascending=False
)

print(df_slopes_incidence)
print()
print(df_slopes_mortality)

En la serie temporal se observa un incremento aparente de las tasas a partir de 2017, que no debe interpretarse como un aumento abrupto de la incidencia real del cáncer. Este comportamiento coincide con actualizaciones metodológicas introducidas por GLOBOCAN, así como con mejoras en los sistemas de diagnóstico y registro en numerosos países, lo que puede generar discontinuidades en las series temporales.


Por otro lado, El análisis principal durante esta primera exploración se centró en la incidencia del cáncer, utilizando tasas estandarizadas por edad (ASR World), y se realizó de forma separada por sexo con el fin de preservar las diferencias epidemiológicas observadas entre hombres y mujeres.

A partir del análisis exploratorio visual, se podría seleccionar el  año 2022 como periodo de referencia, al tratarse del año más reciente y menos afectado por las distorsiones observadas durante la pandemia de COVID-19. Sin embargo, primeramente se debe ver si merece la pena usar ese año o usar otro.





In [ ]:
idx = (
    df_merged
    .groupby(["Country_harmonized", "Sex", "Type"])["Year"]
    .idxmax()
)

df_latest = df_merged.loc[idx].reset_index(drop=True)


In [ ]:
df_plot = df_merged.copy()


for t in df_plot["Type"].unique():
    plt.figure(figsize=(10,5))

    df_plot[df_plot["Type"] == t].boxplot(
        column="ASR (World)",
        by="Year",
        grid=False,
        rot=90
    )

    plt.title(f"ASR (World) distribución por país – {t}")
    plt.suptitle("")
    plt.xlabel("Country")
    plt.ylabel("ASR (World)")
    plt.tight_layout()
    plt.savefig(f"{fig_dir}/fig02_boxplot_ASR_{t}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

stats_simple = (
    df_plot
    .groupby(["Type", "Year"])["ASR (World)"]
    .describe()
)

print(stats_simple)



Estas figuras muestran la distribución de las tasas de incidencia y mortalidad estandarizadas por edad (ASR World) para el periodo 2007–2023 mediante diagramas de caja y bigotes. En ambos casos se observa una elevada heterogeneidad entre países, reflejada en amplios rangos intercuartílicos y la presencia sistemática de valores atípicos. La incidencia presenta una mayor dispersión que la mortalidad, lo que pone de manifiesto su mayor sensibilidad a factores como el diagnóstico precoz y la calidad de los registros. Por el contrario, la mortalidad muestra una distribución más estable y concentrada, lo que sugiere una mayor dependencia de la capacidad terapéutica de los sistemas sanitarios. Estas observaciones refuerzan la utilización de la mortalidad como variable principal, complementada por la incidencia.

Para continuar con el análisis estadístico y valorar que año y paises utilizar en nuestro trabajo, se busca entonces valorar que paises tienen los datos de incidencia y/o mortalidad.



In [ ]:
#cuantos paises hay con año, tipo de cancer y tipo

counts = (
    df_merged
    .groupby(
        ["Country_harmonized", "Year", "Cancer label", "Type"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

 # cuantos países tienen los datos completos (both) o incompletos?

counts["both"] = (
    (counts.get("Incidence", 0) > 0) &
    (counts.get("Mortality", 0) > 0)
)

counts["only_incidence"] = (
    (counts.get("Incidence", 0) > 0) &
    (counts.get("Mortality", 0) == 0)
)

counts["only_mortality"] = (
    (counts.get("Incidence", 0) == 0) &
    (counts.get("Mortality", 0) > 0)
)

summary = {
    "Total combinaciones país–año–cáncer": len(counts),
    "Con incidencia y mortalidad": counts["both"].sum(),
    "Solo incidencia": counts["only_incidence"].sum(),
    "Solo mortalidad": counts["only_mortality"].sum(),
}
print(
    f"Total combinaciones país–año–cáncer: {summary['Total combinaciones país–año–cáncer']}\n"
    f"Con incidencia y mortalidad: {summary['Con incidencia y mortalidad']}\n"
    f"Solo incidencia: {summary['Solo incidencia']}\n"
    f"Solo mortalidad: {summary['Solo mortalidad']}"
)


#Un resumen por año

by_year = (
    counts
    .groupby("Year")[["both", "only_incidence", "only_mortality"]]
    .sum()
)

print(by_year)

#resumen por pais

by_country = (
    counts
    .groupby("Country_harmonized")[["both", "only_incidence", "only_mortality"]]
    .sum()
    .sort_values("both")
)





In [ ]:
plt.figure(figsize=(10, 6))

plt.bar(
    by_year.index,
    by_year["both"],
    label="Incidence & Mortality",
    color="#4C72B0"
)

plt.bar(
    by_year.index,
    by_year["only_incidence"],
    bottom=by_year["both"],
    label="Only incidence",
    color="#DD8452"
)

plt.bar(
    by_year.index,
    by_year["only_mortality"],
    bottom=by_year["both"] + by_year["only_incidence"],
    label="Only mortality",
    color="#55A868"
)

plt.xlabel("Year")
plt.ylabel("Number of country–cancer records")
plt.title("Availability of incidence and mortality data by year")
plt.legend()
plt.tight_layout()
plt.xticks(by_year.index.astype(int))


plt.show()


In [ ]:
by_year_pct = by_year.div(by_year.sum(axis=1), axis=0)

by_year_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    colormap="tab20"
)

plt.xlabel("Year")
plt.ylabel("Proportion of records")
plt.title("Proportion of incidence and mortality data availability by year")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


Se observó una reducción generalizada en el número de países con datos disponibles a partir de 2018 para todos los tipos de cáncer analizados. Este cambio de cobertura limita la comparabilidad temporal de las medias anuales posteriores a dicho año, por lo que los análisis longitudinales se interpretan con cautela y se prioriza el uso de años con alta cobertura, concretamente 2018 y 2022.

Antes de integrar los datos de infraestructura sanitaria, se realizó un análisis exploratorio de la disponibilidad de los registros de incidencia y mortalidad por cáncer por país, año y tipo de cáncer, con el objetivo de evaluar la consistencia temporal y la comparabilidad de ambos indicadores. Este análisis permitió identificar limitaciones en la cobertura conjunta de incidencia y mortalidad a lo largo del tiempo.

A partir de 2017, los datos de incidencia muestran una cobertura insuficiente y heterogénea entre países y tipos de cáncer, lo que limita su uso en análisis comparativos globales. En contraste, la mortalidad mantiene una cobertura amplia y estable, por lo que se adopta como variable principal del estudio.

En consecuencia, el análisis se centra fundamentalmente en la mortalidad por cáncer. La incidencia se utiliza de forma exploratoria en los periodos en los que la disponibilidad de ambos indicadores lo permite, prestando especial atención a los años 2018 y 2022, que se evaluarán como posibles puntos de referencia para análisis comparativos transversales.

El análisis exploratorio de la cobertura geográfica por año mostró una estabilidad de datos incidencia y mortalidad elevada hasta 2016, seguida de una reducción progresiva en los años más recientes. Por este motivo, se busca además la cantidad de datos que se tienen por mortalidad compartiendo o no incidencia al mismo tiempo.
Esta estrategia permite evaluar la robustez de los resultados y analizar posibles cambios temporales sin comprometer la representatividad.

In [ ]:
#Esto por ahora no aporta nada

df_mort_f = df_plot[(df_plot["Type"] == "Mortality") & (df_plot["Sex"] == "Female")]

top5_worst_f = (
    df_mort_f
    .sort_values(["Year", "ASR (World)"], ascending=[True, False])
    .groupby("Year")
    .head(5)
    .reset_index(drop=True)
)

#print(top5_worst_f)


top5_best_f = (
    df_mort_f
    .sort_values(["Year", "ASR (World)"], ascending=[True, True])
    .groupby("Year")
    .head(5)
    .reset_index(drop=True)
)
#print(top5_best_f)

df_mort_m = df_plot[(df_plot["Type"] == "Mortality") & (df_plot["Sex"] == "Male")]


top5_worst_m = (
    df_mort_m
    .sort_values(["Year", "ASR (World)"], ascending=[True, False])
    .groupby("Year")
    .head(5)
    .reset_index(drop=True)
)

#print(top5_worst_m)


top5_best_m = (
    df_mort_m
    .sort_values(["Year", "ASR (World)"], ascending=[True, True])
    .groupby("Year")
    .head(5)
    .reset_index(drop=True)
)

#print(top5_best_m)

In [ ]:
# Ahora los extremos, mujer
summary_extremes_f = (
    df_mort_f
    .groupby("Year")
    .apply(lambda x: pd.DataFrame({
        "Worst_country": x.sort_values("ASR (World)", ascending=False)
                           .head(5)["Country_harmonized"].values,
        "Worst_cancer": x.sort_values("ASR (World)", ascending=False)
                          .head(5)["Cancer label"].values,
        "Worst_ASR": x.sort_values("ASR (World)", ascending=False)
                       .head(5)["ASR (World)"].values,
        "Best_country": x.sort_values("ASR (World)")
                          .head(5)["Country_harmonized"].values,
        "Best_cancer": x.sort_values("ASR (World)")
                         .head(5)["Cancer label"].values,
        "Best_ASR": x.sort_values("ASR (World)")
                      .head(5)["ASR (World)"].values
    }))
)


#print(summary_extremes_f)

summary_extremes_m = (
    df_mort_m
    .groupby("Year")
    .apply(lambda x: pd.DataFrame({
        "Worst_country": x.sort_values("ASR (World)", ascending=False)
                           .head(5)["Country_harmonized"].values,
        "Worst_cancer": x.sort_values("ASR (World)", ascending=False)
                          .head(5)["Cancer label"].values,
        "Worst_ASR": x.sort_values("ASR (World)", ascending=False)
                       .head(5)["ASR (World)"].values,
        "Best_country": x.sort_values("ASR (World)")
                          .head(5)["Country_harmonized"].values,
        "Best_cancer": x.sort_values("ASR (World)")
                         .head(5)["Cancer label"].values,
        "Best_ASR": x.sort_values("ASR (World)")
                      .head(5)["ASR (World)"].values
    }))
)

#print(summary_extremes_m)

In [ ]:
  #calcula las medias de mortalidd tanto de hombres como mujeres
mean_mort_m_by_year = (
    df_mort_m
    .groupby("Year")["ASR (World)"]
    .mean()
)


mean_mort_f_by_year = (
    df_mort_f
    .groupby("Year")["ASR (World)"]
    .mean()
)




#calculas los mejores paises con baja/alta mortalidad segun sexo

top5_best_16_22_m = (
    df_mort_m[df_mort_m["Year"].isin([2016, 2022])].copy()
    .sort_values("ASR (World)", ascending=True)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_worst_16_22_m = (
    df_mort_m[df_mort_m["Year"].isin([2016, 2022])].copy()
    .sort_values("ASR (World)", ascending=False)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_best_16_22_f = (
    df_mort_f[df_mort_f["Year"].isin([2016, 2022])].copy()
    .sort_values("ASR (World)", ascending=True)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_worst_16_22_f = (
    df_mort_f[df_mort_f["Year"].isin([2016, 2022])].copy()
    .sort_values("ASR (World)", ascending=False)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)



In [ ]:
#ahora con 2018-2022
#calcula las medias de mortalidd tanto de hombres como mujeres
mean_mort_m_by_year = (
    df_mort_m
    .groupby("Year")["ASR (World)"]
    .mean()
)


mean_mort_f_by_year = (
    df_mort_f
    .groupby("Year")["ASR (World)"]
    .mean()
)




#calculas los mejores paises con baja/alta mortalidad segun sexo

top5_best_18_22_m = (
    df_mort_m[df_mort_m["Year"].isin([2018, 2022])].copy()
    .sort_values("ASR (World)", ascending=True)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_worst_18_22_m = (
    df_mort_m[df_mort_m["Year"].isin([2018, 2022])].copy()
    .sort_values("ASR (World)", ascending=False)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_best_18_22_f = (
    df_mort_f[df_mort_f["Year"].isin([2018, 2022])].copy()
    .sort_values("ASR (World)", ascending=True)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_worst_18_22_f = (
    df_mort_f[df_mort_f["Year"].isin([2018, 2022])].copy()
    .sort_values("ASR (World)", ascending=False)
    .groupby("Year", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)



In [ ]:
def plot_top5_by_sex_same_height(df_m, df_f, year, title_prefix, fig_dir=None, fname=None):


    # Ordenar ambos por severidad (de menor a mayor para que el peor quede arriba)
    df_m = df_m.sort_values("ASR (World)", ascending=True).reset_index(drop=True)
    df_f = df_f.sort_values("ASR (World)", ascending=True).reset_index(drop=True)


    # Etiquetas por sexo
    labels_m = df_m["Country_harmonized"] + " (" + df_m["Cancer label"] + ")"
    labels_f = df_f["Country_harmonized"] + " (" + df_f["Cancer label"] + ")"

    # Posiciones comunes
    y = np.arange(len(df_m))  # Top 5 → 0..4

    fig, ax = plt.subplots(figsize=(10, 6))

    # HOMBRES (izquierda)
    ax.barh(
        y,
        -df_m["ASR (World)"],
        color="#003f5c",
        height=0.6,
        label="Men"
    )

    # MUJERES (derecha)
    ax.barh(
        y,
        df_f["ASR (World)"],
        color="#F4A6C1",
        height=0.6,
        label="Women"
    )

    # Línea central
    ax.axvline(0, color="black", linewidth=1)
    #dibujar los verticales de las medias por sexo
      # Línea media hombres (izquierda, negativa)
    ax.axvline(
      -mean_mort_m_by_year.loc[year],
      color="#003f5c",
      linestyle="--",
      linewidth=2,
      alpha=0.8,
      label=f"Mean men: {mean_mort_m_by_year[year]:.2f}"
  )

# Línea media mujeres (derecha, positiva)
    ax.axvline(
      mean_mort_f_by_year.loc[year],
      color="#F4A6C1",
      linestyle="--",
      linewidth=2,
      alpha=0.8,
      label=f"Mean women: {mean_mort_f_by_year.loc[year]:.2f}"
  )


    # Eje Y izquierdo → hombres
    ax.set_yticks(y)
    ax.set_yticklabels(labels_m)

    # Eje Y derecho → mujeres
    ax_r = ax.twinx()
    ax_r.set_yticks(y)
    ax_r.set_yticklabels(labels_f)
    ax_r.set_ylim(ax.get_ylim())

    # Escalas y títulos
    max_val = max(
        df_m["ASR (World)"].max(),
        df_f["ASR (World)"].max()
    ) * 1.2

    ax.set_xlim(-max_val, max_val)
    ax.set_xlabel("Mortality ASR (World)")
    ax.set_title(f"{title_prefix} – {year} (Top 5 per sex)")
    ax.legend(loc="lower right")

    plt.tight_layout()
    if fig_dir and fname:
        plt.savefig(f"{fig_dir}/{fname}", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
#Para los peores resultados en mortalidad

for year in [2016, 2022]:
    df_m = top5_worst_16_22_m[top5_worst_16_22_m["Year"] == year].head(5)
    df_f = top5_worst_16_22_f[top5_worst_16_22_f["Year"] == year].head(5)

    plot_top5_by_sex_same_height(
        df_m, df_f, year,
        title_prefix="Worst mortality",
        fig_dir=fig_dir,
        fname=f"worst_mortality_top5_by_sex_{year}.jpg"
    )

#par los mejores resultados en mortalidad
for year in [2016, 2022]:
    df_m = top5_best_16_22_m[top5_best_16_22_m["Year"] == year].head(5)
    df_f = top5_best_16_22_f[top5_best_16_22_f["Year"] == year].head(5)

    plot_top5_by_sex_same_height(
        df_m, df_f, year,
        title_prefix="Best mortality",
        fig_dir=fig_dir,
        fname=f"best_mortality_top5_by_sex_1622{year}.jpg"
    )


In [ ]:
#cpm 2018
#Para los peores resultados en mortalidad

for year in [2018, 2022]:
    df_m = top5_worst_18_22_m[top5_worst_18_22_m["Year"] == year].head(5)
    df_f = top5_worst_18_22_f[top5_worst_18_22_f["Year"] == year].head(5)

    plot_top5_by_sex_same_height(
        df_m, df_f, year,
        title_prefix="Worst mortality",
        fig_dir=fig_dir,
        fname=f"worst_mortality_top5_by_sex_{year}.jpg"
    )

#par los mejores resultados en mortalidad
for year in [2018, 2022]:
    df_m = top5_best_18_22_m[top5_best_18_22_m["Year"] == year].head(5)
    df_f = top5_best_18_22_f[top5_best_18_22_f["Year"] == year].head(5)

    plot_top5_by_sex_same_height(
        df_m, df_f, year,
        title_prefix="Best mortality",
        fig_dir=fig_dir,
        fname=f"best_mortality_top5_by_sex_1822{year}.jpg"
    )


La mortalidad media se calculó de forma independiente para cada año y sexo, evitando mezclar periodos temporales con distinta cobertura y garantizando la comparabilidad entre 2016 y 2022, y 2018-22.

Las líneas discontinuas representan la mortalidad media (ASR World) de los  países registrado para los años 2016 y 2022, y 2018-22 respectivamente.
 Se observa que la mortalidad media masculina es sistemáticamente superior a la femenina, tanto en magnitud como en dispersión, lo que refuerza la necesidad de realizar análisis estratificados por sexo.

Para saber hasta donde llegan nuestro alcance con los datos si se consideran tanto incidencia como mortalidad, graficamos el número de países registrados por año

In [ ]:
#cada tipo de cancer, un color distinto

cancer_colors = {
    "Lung": "#4C72B0",        # azul
    "Breast": "#DD8452",      # naranja
    "Prostate": "#55A868",    # verde
    "Colon": "#C44E52",       # rojo
    "Cervix": "#8172B2",      # morado
    "Lip_oc_Pharynx": "#CCB974",  # amarillo
}

#Preparar por año y sexo la mortalidad para agrupar por pais y tipo de cancer
def prepare_stacked_mortality_by_sex(df, year, sex):
    df_y = df[
        (df["Year"] == year) &
        (df["Type"] == "Mortality") &
        (df["Sex"] == sex)
    ].copy()

    agg = (
        df_y
        .groupby(["Country_harmonized", "Cancer label"], as_index=False)
        .agg({"ASR (World)": "mean"})
    )

    return agg


def pivot_stacked(df_agg):
    pivot = df_agg.pivot(
        index="Country_harmonized",
        columns="Cancer label",
        values="ASR (World)"
    ).fillna(0)

    return pivot

#función para graficar las barras por país la mortalidad
def plot_stacked_mortality_by_sex(pivot, year, sex_label, fig_dir=None):
    plt.figure(figsize=(14, 6))

    bottom = np.zeros(len(pivot))

    for cancer in pivot.columns:
        plt.bar(
            pivot.index,
            pivot[cancer],
            bottom=bottom,
            label=cancer,
            color=cancer_colors[cancer] # Use cancer to get color dynamically
        )
        bottom += pivot[cancer].values

    plt.xticks(rotation=90)
    plt.ylabel("Mortality ASR (World)")
    plt.title(f"Mortality by cancer type – {sex_label} – {year}")
    plt.legend(title="Cancer type", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()

    if fig_dir:
        plt.savefig(
            f"{fig_dir}/stacked_mortality_{sex_label}_{year}.jpg",
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()



# 2016

#Para hombres
mort_2016_m = prepare_stacked_mortality_by_sex(df_merged, 2016, "Male")



pivot_2016_m_full = pivot_stacked(mort_2016_m)
existing_cancer_keys_m = [key for key in cancer_colors.keys() if key in pivot_2016_m_full.columns]
pivot_2016_m = pivot_2016_m_full[existing_cancer_keys_m]

pivot_2016_m = pivot_2016_m.loc[
    pivot_2016_m.sum(axis=1).sort_values(ascending=False).index
]

#para mujeres
mort_2016_f = prepare_stacked_mortality_by_sex(df_merged, 2016, "Female")
pivot_2016_f_full = pivot_stacked(mort_2016_f)
existing_cancer_keys_f = [key for key in cancer_colors.keys() if key in pivot_2016_f_full.columns]
pivot_2016_f = pivot_2016_f_full[existing_cancer_keys_f]

pivot_2016_f = pivot_2016_f.loc[
    pivot_2016_f.sum(axis=1).sort_values(ascending=False).index
]


plot_stacked_mortality_by_sex(
    pivot_2016_m,
    2016,
    "Men",
    fig_dir
)

plot_stacked_mortality_by_sex(
    pivot_2016_f,
    2016,
    "Women",
    fig_dir
)


#2018

#Para hombres
mort_2018_m = prepare_stacked_mortality_by_sex(df_merged, 2018, "Male")



pivot_2018_m_full = pivot_stacked(mort_2018_m)
existing_cancer_keys_m = [key for key in cancer_colors.keys() if key in pivot_2018_m_full.columns]
pivot_2018_m = pivot_2018_m_full[existing_cancer_keys_m]

pivot_2018_m = pivot_2018_m.loc[
    pivot_2018_m.sum(axis=1).sort_values(ascending=False).index
]

#para mujeres
mort_2018_f = prepare_stacked_mortality_by_sex(df_merged, 2018, "Female")
pivot_2018_f_full = pivot_stacked(mort_2018_f)
existing_cancer_keys_f = [key for key in cancer_colors.keys() if key in pivot_2018_f_full.columns]
pivot_2018_f = pivot_2018_f_full[existing_cancer_keys_f]

pivot_2018_f = pivot_2018_f.loc[
    pivot_2018_f.sum(axis=1).sort_values(ascending=False).index
]


plot_stacked_mortality_by_sex(
    pivot_2018_m,
    2018,
    "Men",
    fig_dir
)

plot_stacked_mortality_by_sex(
    pivot_2018_f,
    2018,
    "Women",
    fig_dir
)


# 2022
#graficar para hombres
mort_2022_m = prepare_stacked_mortality_by_sex(df_merged, 2022, "Male")

pivot_2022_m_full = pivot_stacked(mort_2022_m)
existing_cancer_keys_m_2022 = [key for key in cancer_colors.keys() if key in pivot_2022_m_full.columns]
pivot_2022_m = pivot_2022_m_full[existing_cancer_keys_m_2022]

pivot_2022_m = pivot_2022_m.loc[
    pivot_2022_m.sum(axis=1).sort_values(ascending=False).index
]

#graficar Para mujeres
mort_2022_f = prepare_stacked_mortality_by_sex(df_merged, 2022, "Female")
pivot_2022_f_full = pivot_stacked(mort_2022_f)
existing_cancer_keys_f_2022 = [key for key in cancer_colors.keys() if key in pivot_2022_f_full.columns]
pivot_2022_f = pivot_2022_f_full[existing_cancer_keys_f_2022]

pivot_2022_f = pivot_2022_f.loc[
    pivot_2022_f.sum(axis=1).sort_values(ascending=False).index
]


plot_stacked_mortality_by_sex(
    pivot_2022_m,
    2022,
    "Men",
    fig_dir
)

plot_stacked_mortality_by_sex(
    pivot_2022_f,
    2022,
    "Women",
    fig_dir
)

In [ ]:
df_merged.head()

In [ ]:
df_2016 = df_merged[df_merged["Year"] == 2016]
df_2018 = df_merged[df_merged["Year"] == 2018]
df_2022 = df_merged[df_merged["Year"] == 2022]

print("El numero de paises registrados en el 2016:", df_2016["Country_harmonized"].nunique())
print("El numero de paises registrados en el 2018:", df_2018["Country_harmonized"].nunique())
print("El número de países registrados en el 2022:" , df_2022["Country_harmonized"].nunique())


In [ ]:
df_merged["Last Update"].value_counts()


A continuación se hará un primer análisis sobre incidencia y mortalidad.


In [ ]:
complete_keys = (
    df_merged
    .groupby(["Country_harmonized", "Year", "Sex"])["Type"]
    .nunique()
    .reset_index()
    .query("Type == 2")
)


df_complete = df_merged.merge(
    complete_keys[["Country_harmonized", "Year", "Sex"]],
    on=["Country_harmonized", "Year", "Sex"],
    how="inner"
)
#Ahora solo contiene observaciones donde existen ambos tipos.

def build_pivot(df):
    df_scatter = (
        df
        .groupby(
            ["Country_harmonized", "Year", "Type"],
            as_index=False
        )
        .agg({"ASR (World)": "mean"})
    )

    pivot = (
        df_scatter
        .pivot_table(
            index=["Country_harmonized", "Year"],
            columns="Type",
            values="ASR (World)"
        )
        .reset_index()
    )

    return pivot

pivot_male = build_pivot(df_male)
pivot_female = build_pivot(df_female)
pivot_all = build_pivot(df_complete)

pivot_male_clean = pivot_male.dropna(subset=["Incidence", "Mortality"])
pivot_female_clean = pivot_female.dropna(subset=["Incidence", "Mortality"])


print("Male:", pivot_male_clean.shape)
print("Female:", pivot_female_clean.shape)

print(
    "Male - países:",
    pivot_male_clean["Country_harmonized"].nunique()
)

print(
    "Female - países:",
    pivot_female_clean["Country_harmonized"].nunique()
)

In [ ]:
def compute_beta_R(df):
    x = df["Incidence"].values
    y = df["Mortality"].values

    beta, alpha = np.polyfit(x, y, 1)
    R = np.corrcoef(x, y)[0, 1]

    return beta, alpha, R

beta_f, alpha_f, R_f = compute_beta_R(pivot_female_clean)
beta_m,  alpha_m,  R_m  = compute_beta_R(pivot_male_clean)
beta_all, alpha_all, R_all = compute_beta_R(pivot_all)

print(f"Female  -> beta={beta_f:.3f}, alpha={alpha_f:.3f}, R={R_f:.3f}, N={len(pivot_female_clean)}")
print(pivot_female_clean.head())

print(f"Male    -> beta={beta_m:.3f},  alpha={alpha_m:.3f},  R={R_m:.3f},  N={len(pivot_male_clean)}")
print(pivot_male_clean.head())

print(f"All    -> beta={beta_all:.3f},  alpha={alpha_all:.3f},  R={R_all:.3f},  N={len(pivot_all)}")
print(pivot_all.head())

Al estratificar el análisis por sexo, se observan diferencias claras en la relación entre incidencia y mortalidad. En hombres, la pendiente del modelo es mayor, lo que indica un incremento más acusado de la mortalidad a medida que aumenta la incidencia. En mujeres, la relación es menos pronunciada pero más consistente, reflejada en un coeficiente de correlación superior. Por el contrario, el análisis agregado por sexo presenta una menor capacidad explicativa, lo que evidencia que la agregación de ambos sexos introduce heterogeneidad y enmascara patrones relevantes





In [ ]:
# cálculo de la recta en mujeres
plt.figure(figsize=(7, 7))

years = sorted(pivot_female_clean["Year"].unique())
cmap = plt.cm.viridis
colors = cmap(np.linspace(0, 1, len(years)))

for y, c in zip(years, colors):
    df_y = pivot_female_clean[pivot_female_clean["Year"] == y]

    plt.scatter(
        df_y["Incidence"],
        df_y["Mortality"],
        alpha=0.6,
        color=c,
        label=str(y)
    )

# Recta global (
x_f = pivot_female_clean["Incidence"].values
y_f = pivot_female_clean["Mortality"].values
beta_f, alpha_f = np.polyfit(x_f, y_f, 1)

x_line = np.linspace(x_f.min(), x_f.max(), 200)
plt.plot(x_line, beta_f * x_line + alpha_f, linestyle="--", color="black")

plt.xlabel("Incidence ASR (World)")
plt.ylabel("Mortality ASR (World)")
plt.title("Female – Mortality vs Incidence (ASR World)\nColored by Year")

plt.legend(
    title="Year",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.gca().set_aspect("equal", adjustable="box")
plt.grid(False)
plt.tight_layout()
plt.show()



Esta gráfica nos indica que a medida que se incrementa la incidencia, la mortalidad también incremente, sin embargo, al ver tanta dispersión vertical, lo que puede sugerir es que el riesgo de morir no solamente depende de cuantos casos hay, sino de factores como acceso al tratamientos, detección precoz, infraestructura (radioterapia), pib per cápita, etc. Al ver el comportamiento a lo largo del tiempo, se muestra que la relación incidencia-mortalidad es estructural.

In [ ]:
#hombres


plt.figure(figsize=(7, 7))

years = sorted(pivot_male_clean["Year"].unique())
cmap = plt.cm.viridis
colors = cmap(np.linspace(0, 1, len(years)))

for y, c in zip(years, colors):
    df_y = pivot_male_clean[pivot_male_clean["Year"] == y]

    plt.scatter(
        df_y["Incidence"],
        df_y["Mortality"],
        alpha=0.6,
        color=c,
        label=str(y)
    )

# Recta global
x_m = pivot_male_clean["Incidence"].values
y_m = pivot_male_clean["Mortality"].values
beta_m, alpha_m = np.polyfit(x_m, y_m, 1)

x_line = np.linspace(x_m.min(), x_m.max(), 200)
plt.plot(x_line, beta_m * x_line + alpha_m, linestyle="--", color="black")

plt.xlabel("Incidence ASR (World)")
plt.ylabel("Mortality ASR (World)")
plt.title("Men – Mortality vs Incidence (ASR World)\nColored by Year")

plt.legend(
    title="Year",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.gca().set_aspect("equal", adjustable="box")
plt.grid(False)
plt.tight_layout()
plt.show()



En este gráfico se puede ver cuan diferente es si se grafica lo mismo incidencia-mortalidad pero en hombres, donde la pendiente es más pronunciada y presenta mayor dispersión vertical. Dando lugar a preguntas como, ¿que ocurre con el acceso a los tratamientos? y en nuestro caso, a las instalaciones de radioterapia.

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    x_f, y_f,
    alpha=0.5,
    label=f"Women (R={R_f:.2f})"
)

plt.scatter(
    x_m, y_m,
    alpha=0.5,
    label=f"Men (R={R_m:.2f})"
)

# rectas
x_min = min(x_f.min(), x_m.min())
x_max = max(x_f.max(), x_m.max())
x_line = np.linspace(x_min, x_max, 200)

plt.plot(x_line, beta_f * x_line + alpha_f, linestyle="--")
plt.plot(x_line, beta_m * x_line + alpha_m, linestyle="--")

plt.xlabel("Incidence ASR (World)")
plt.ylabel("Mortality ASR (World)")
plt.title("Mortality vs Incidence (ASR World)\nStratified by Sex")

plt.legend()
plt.gca().set_aspect("equal", adjustable="box")
plt.grid(False)
plt.tight_layout()
plt.show()


Las representaciones gráficas de la relación entre incidencia y mortalidad estandarizadas por edad revelan patrones claramente diferenciados por sexo. En el caso de los hombres, se observa una pendiente más pronunciada y una mayor dispersión de los datos, lo que indica que incrementos en la incidencia se asocian con aumentos más acusados de la mortalidad. En las mujeres, la relación es más moderada y presenta una mayor consistencia, de modo que la mortalidad aumenta de forma más gradual con la incidencia.

Los coeficientes de correlación observados (R ≈ 0.60 en hombres y R ≈ 0.63 en mujeres) indican una asociación positiva de magnitud moderada en ambos sexos, lo que sugiere que la incidencia explica solo parcialmente la variabilidad en la mortalidad. La superposición de ambos grupos en un análisis agregado oculta estas diferencias estructurales, lo que justifica la estratificación por sexo adoptada en este estudio.

Asimismo, la mayor pendiente y dispersión observadas en los hombres apuntan a una relación más intensa entre incidencia y mortalidad, potencialmente relacionada con diferencias en los tipos de cáncer predominantes, los perfiles de riesgo y el acceso al diagnóstico precoz y al tratamiento. Estos resultados refuerzan la necesidad de ampliar el análisis incorporando variables adicionales, como el PIB per cápita y la disponibilidad de equipos de radioterapia, con el fin de explorar factores explicativos de las diferencias observadas en la mortalidad.



Justificación del uso de la mortalidad como variable de resultado (2016–2022)

Con el objetivo de garantizar la comparabilidad temporal y maximizar la cobertura geográfica del análisis, se evaluó la disponibilidad de datos de incidencia y mortalidad para los años 2016, 2018 y 2022, que constituyen los dos puntos temporales de referencia en este estudio.

El año 2016 representa el último periodo en el que una proporción sustancial de países dispone simultáneamente de datos de incidencia y mortalidad, permitiendo una caracterización relativamente completa de ambos indicadores. Sin embargo, en 2022 se observa un cambio estructural muy marcado en la disponibilidad de los datos: la mayoría de los registros país–cáncer corresponden exclusivamente a mortalidad, mientras que los datos de incidencia están disponibles únicamente para un número reducido de países.

Este contraste implica que el uso de la incidencia, o de modelos que requieran simultáneamente incidencia y mortalidad, conduciría en 2022 a una reducción drástica del tamaño muestral y a un sesgo hacia países con sistemas de registro oncológico más desarrollados. En cambio, la mortalidad mantiene una cobertura geográfica amplia y homogénea en ambos años, permitiendo una comparación consistente entre países y a lo largo del tiempo.

Dado que el objetivo del trabajo es analizar la relación entre los resultados en cáncer y la disponibilidad de infraestructuras de radioterapia, se selecciona la mortalidad como variable principal utilizando el año 2022. Esta elección permite maximizar el número de países incluidos, garantizar la coherencia temporal del análisis y centrarse en un desenlace clínico final estrechamente vinculado al acceso y la efectividad de los tratamientos oncológicos.

Se realizará lo mismo con el año 2018.



###DIRAC

A partir de esta evaluación, los análisis posteriores se centran exclusivamente en la mortalidad por cáncer, que se relaciona con la disponibilidad de infraestructura de radioterapia (DIRAC) y se ajusta por población.

Ahora, se construye el primer scatter con indice de mortalidad vs centros de radioterapia existentes. Para esto, es necesario normalizar la infraestructura por población, es decir, RT centers y equipos por millón de habitantes.

In [ ]:
df_merged_all = df_merged.copy()

# Quedarse solo con mortalidad
df_merged_all = df_merged_all[df_merged_all["Type"] == "Mortality"].copy()

# Ver qué años hay y cuántas observaciones por año
df_merged_all["Year"].value_counts().sort_index()


In [ ]:
# 1. Renombrar columnas correctamente
rt_vars = ["RTCenters", "Linac", "Protontherapy", "XRay", "Brachytherapy"]

for var in rt_vars:
    df_merged_all[f"{var}_m"] = (
        df_merged_all[var] / df_merged_all["Population"]
    ) * 1_000_000

# 2. Calcular LINACs necesarios SIN duplicar por sexo
linacs_needed = (
    df_merged_all
    .groupby(['Country_harmonized', 'Year'], as_index=False)
    .agg({'Population': 'first', 'Linac': 'first'})
)

#En la literatura técnica y en estudios revisados por pares, se utiliza
#comúnmente como referencia un benchmark de aproximadamente una máquina de
#radioterapia de megavoltaje por cada 250 000 habitantes, atribuido a las
#directrices de la IAEA para planificación de servicios de radioterapia.
#Balogun 2016

linacs_needed['rt_linac_needed'] = linacs_needed['Population'] / 250_000
linacs_needed['rt_deficit'] = (
    linacs_needed['rt_linac_needed'] - linacs_needed['Linac']
)
linacs_needed['coverage_index'] = (
    linacs_needed['Linac'] / linacs_needed['rt_linac_needed']
) * 100

df_merged_all = df_merged_all.merge(
    linacs_needed[['Country_harmonized', 'Year', 'rt_linac_needed',
                   'rt_deficit', 'coverage_index']],
    on=['Country_harmonized', 'Year'],
    how='left'
)

# 3. Guardar dataset limpio
df_merged_all.to_csv("df_merged_clean.csv", index=False)

En la literatura técnica y en estudios revisados por pares, se utiliza comúnmente como referencia un benchmark de aproximadamente una máquina de radioterapia de megavoltaje por cada 250 000 habitantes, atribuido a las directrices de la IAEA para planificación de servicios de radioterapia.


In [ ]:

df_merged_all.head()

In [ ]:
df_merged_all["Cancer label"].unique()
df_merged_all["Cancer label"].value_counts()

In [ ]:
df_merged_all["Last Update"].value_counts()


Se grafican los valores dispersos de mortalidad y equipos de radioterapias varios, empezando con búsquedas (sin mucha pretención) de comportamientos lineales.
Primer modelo lineal:

In [ ]:
#definimos el primer modelo

df_model = df_merged_all[
    (df_merged_all["Year"].isin([2018, 2022])) &
    (df_merged_all["Type"] == "Mortality")
].copy()



In [ ]:
#dispersión por sexo y cancer
def scatter_by_sex_cancer_year(
    df, cancer, sex, rt_var_pm, years=[2018, 2022]
):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, year in zip(axes, years):
        df_y = df[
            (df["Cancer label"] == cancer) &
            (df["Sex"] == sex) &
            (df["Year"] == year)
        ]

        ax.scatter(
            df_y[rt_var_pm],
            df_y["ASR (World)"],
            alpha=0.7
        )

        ax.set_title(f"{year}")
        ax.set_xlabel(
            rt_var_pm.replace("_m", "")
            + "\n(per million inhabitants)"
        )

    axes[0].set_ylabel("Mortality ASR (World)")
    plt.suptitle(f"{cancer} – {sex}")
    plt.tight_layout()
    plt.show()

In [ ]:
scatter_by_sex_cancer_year(
    df=df_model,
    cancer="Lip_oc_Pharynx",
    sex="Male",
    rt_var_pm="Linac_m"
)

In [ ]:
scatter_by_sex_cancer_year(
    df_model,
    cancer="Cervix",
    sex="Female",
    rt_var_pm="Brachytherapy_m"
)


In [ ]:
import numpy as np
from scipy import stats

# Datos ejemplo (aproximados del gráfico)
# 2018
brachy_2018 = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.3, 1.5, 2.0, 2.5, 3.0]
mortality_2018 = [9.2, 6.4, 8.6, 6.0, 5.5, 4.7, 4.3, 2.9, 3.9, 1.5, 6.9]

# 2022
brachy_2022 = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.3, 1.5, 2.0, 2.5, 3.0]
mortality_2022 = [5.4, 4.4, 7.6, 5.4, 3.5, 4.6, 3.9, 2.5, 3.5, 1.4, 5.6]

# Correlación de Pearson
corr_2018, p_2018 = stats.pearsonr(brachy_2018, mortality_2018)
corr_2022, p_2022 = stats.pearsonr(brachy_2022, mortality_2022)

print(f"Correlación 2018: r={corr_2018:.3f}, p={p_2018:.3f}")
print(f"Correlación 2022: r={corr_2022:.3f}, p={p_2022:.3f}")

# Regresión lineal
slope_2018, intercept_2018, r_value_2018, p_value_2018, std_err_2018 = \
    stats.linregress(brachy_2018, mortality_2018)

print(f"\nRegresión 2018:")
print(f"  Pendiente: {slope_2018:.3f} (cambio en mortalidad por unidad brachy)")
print(f"  R²: {r_value_2018**2:.3f}")
print(f"  p-value: {p_value_2018:.3f}")

# Análisis por grupos
low_brachy = [m for b, m in zip(brachy_2022, mortality_2022) if b < 1.0]
high_brachy = [m for b, m in zip(brachy_2022, mortality_2022) if b >= 1.0]

print(f"\n2022 - Mortalidad promedio:")
print(f"  Brachy <1.0/millón: {np.mean(low_brachy):.2f} ASR")
print(f"  Brachy ≥1.0/millón: {np.mean(high_brachy):.2f} ASR")
print(f"  Diferencia: {np.mean(low_brachy) - np.mean(high_brachy):.2f} ASR")

# Test estadístico
t_stat, p_value = stats.ttest_ind(low_brachy, high_brachy)
print(f"  t-test p-value: {p_value:.3f}")

El signo negativo (-) indica que a mayor braquiterapia, menor mortalidad (relación esperada)

FUERZA MODERADA: r=-0.568 (2018) indica ~32% de la variabilidad en mortalidad explicada por braquiterapia

NO CONCLUYENTE: p=0.068 está "cerca" de significancia (0.05), pero estadísticamente no podemos rechazar H0 (no relación)

DEBILITAMIENTO: La correlación bajó de -0.568 (2018) a -0.450 (2022)

La disponibilidad de equipos de braquiterapia, muy utilizado en cáncer de cervix, muestra una mejoria/disminución en el índice de mortalidad a medida que aumenta la cantidad de equipos. En paises con menos equipos, mayor es la mortalidad, sin embargo, no posee una significancia concluyente, por lo tanto, no poddemos rechazar H0 la no relación.
Presenta además mucha dispersión vertical.

Esto no es una prueba causal, pero si una señal temporal coherente.

Al pasar de 0 a 1 equipo/millón se asocia a una reducción de la mortalidad bastante significativa, sin embargo, al pasar al doble de equipos, se aplana el efecto, esto podría deberse a pocos rendimientos de la infraestrucura sanitaria.

La infraestructura radioterapica es una variable explicativa poderosa para utilizar durante este trabajo, ahora se buscará cuantos equipos de radioterapia en total hay y si es significativo considerar todos o solamente los más utilizados como la braquiterapia y aceleradores lineales (Linac).

Para ejemplos como el cancer y pulmon y los aceleradores lineales, no existe esa correlación tan evidente, por eso, en este estudio, se analizarán distintos tipos de cancer y varios equipos de radioterapia, así como el PIB per cápita.

## Variable objetivo: mortalidad en ASR(World)

La mortalidad en ASR(World) nos indica la tasa de mortalidad ajustada por edad que resultaría si la población de cada país tuviera la misma estructura de edad que una población estándar mundial.

## Variables Predictoras o explicativas

Como variables predictoras se consideran tanto factores económicos como de infraestructura sanitaria. En particular, se utiliza el PIB per cápita como indicador del nivel de desarrollo económico y de la capacidad general del sistema sanitario. Asimismo, se incluyen los distintos tipos de equipos de radioterapia disponibles por millón de habitantes, con el objetivo de capturar la disponibilidad y capacidad terapéutica en cada país.

Adicionalmente, el análisis se estratifica por tipo de cáncer y sexo, dado que la incidencia, la mortalidad y la respuesta al tratamiento presentan diferencias relevantes según estas dimensiones. Esta estratificación permite evaluar de forma más precisa la heterogeneidad en los patrones de mortalidad y en la relación entre infraestructura radioterápica y resultados en salud.

## Selección temporal y enfoque de modelización

A partir del análisis exploratorio y del proceso de depuración de los datos, se identificaron limitaciones relevantes tanto en la cobertura temporal como en la naturaleza de las relaciones entre las variables de interés. En particular, los análisis de disponibilidad mostraron que, a partir de 2017, los datos de incidencia presentan una cobertura insuficiente y heterogénea entre países y tipos de cáncer, mientras que la mortalidad mantiene una cobertura amplia y consistente. Por este motivo, la mortalidad estandarizada por edad (ASR, World) se adopta como variable objetivo principal del estudio.

Dentro del periodo disponible, el año 2022 se selecciona como año de referencia para el análisis principal, al tratarse del año más reciente con información consolidada y comparativamente homogénea en las variables epidemiológicas y de infraestructura sanitaria. Esta elección permite maximizar la representatividad del análisis y centrar el estudio en una fotografía actualizada de la relación entre mortalidad por cáncer y disponibilidad de recursos tanto terapéuticos como económicos.

Por otra parte, las representaciones gráficas y análisis exploratorios realizados evidencian que la relación entre la mortalidad y las variables explicativas —incluyendo la incidencia y la disponibilidad de equipos de radioterapia— no sigue un patrón lineal simple. En particular, se observan efectos no lineales, asimetrías y posibles rendimientos decrecientes, especialmente en los niveles bajos de infraestructura sanitaria. En este contexto, el uso de modelos lineales convencionales podría resultar restrictivo o conducir a interpretaciones parciales.

En consecuencia, se adopta un enfoque basado en Weight of Evidence (WOE) para la construcción del modelo base.

-----------------------------------------------------

Tras el análisis exploratorio y la depuración de los datos, se selecciona el año 2022 como referencia principal por su mayor cobertura y consistencia.

 Dado que las relaciones observadas entre mortalidad e infraestructuras sanitarias no presentan un comportamiento lineal, se adopta un enfoque basado en Weight of Evidence (WOE) como modelo base, que permite capturar de forma más adecuada la naturaleza no lineal de dichas relaciones.



***


# WOE

 En particular por defecto `optbinning` define el WOE de una categoría $i$ como
$$ WOE_i =  ln \left ( {Non-event_i \over Non-event_{total}} \over {Event_i \over Event_{total}}    \right ) $$

¿bajo que criterio utilizaremos IV? que valor?
Por talmotivo utilizaremos como criterio de selección exclusivamente IV<0.002

Podemos extraer el IV y el índice de Gini a partir de la tabla

# Estimación del Modelo

Ahora podemos calcular la tarjeta de puntuación. En los apuntes de clase definimos tanto los WOE, como los Odd ratio como la probabilidad de `evento` respecto al `no-evento` (malos clientes o impago=1 respecto a los buenos clientes o impago=0):

$$ odd = {{P}\over {(1-P)}} ~~ {,~~  siendo} ~~  {P=Prob(impago=1)} $$

 Y la fórmula para obtener la puntuación o los score debe ser una relación negativa con los odd ratio: cuanto mayor la probabilidad de impago (en relación a la de no impago), menor puntuación ha de tener:

 $$ score {= offset - Factor}~·~{ln(odds)}$$

Para pasar de Probabiliddes de impago a Puntuaciones, habrá que establecer tanto el valor de `offset` como el de `Factor`. Esto se hace de manera arbitraria dependiendo de cada institución financiera.

En general, para determinar estos dos valores es necesario establecer la pendiente de la recta y un punto de la misma.

En cuanto a la pendiente, cuanto más plana sea la pendiente, menor variabilidad tendrán los valores de puntuación de crédito que se alcancen, y al revés, cuanto mayor pendiente más diferencias en la puntuación final. Yo voy a utilizar un apendiente (arbitraria) estableciendo de forma arbitraria cada cuantos puntos de score (**pdo_0**) se dobla el odd ratio: $ score - pdo_0 = {offset -Factor}~ ·{ln(2*odds)}$.

 En cuanto al punto de la recta (arbitrario), puede hacerse estableciento (de manera arbitraria) la puntuación o score considerada como de sobresaliente(**scorecard_points**) y el odd ratio que debería tener ese cliente de *sobresaliente* (**odds_0**)

 Así habría que establecer tres parámetros para transformar probabilidades de impago a puntuaciones, por ejemplo:   

* **pdo_0** =40  (esto es que cada 40 puntos de calidad creditica se dobla el odd-ratio))
* **scorecard_points** =600  (alguien con calidad crediticia muy buena, de sobresaliente, sacaría 600 puntos)
* **odds_0** =1/50  (odd ratio que se considera de sobresaliente)

La librería `optBinning` [librería OptBinning](http://gnpalencia.org/optbinning/), en realidad utiliza el módulo de `credit scoring` de `SAS-miner` como inspiración, y por eso define al revés tanto los WOE como los odd ratio, es decir `no-evento` en relación a `evento` (clientes buenos respecto a los malos, o no-impago respecto a impago, impago=0 respecto a impago=1).
$$ odd^B = {{(1-P)}\over {P}} ~~ {,~~  siendo} ~~  {P=Prob(impago=1)} $$

Esto implica que la ecuación que transforma las probabilidades de impago en scores utilizando esta *odds<sup>B</sup>* debe tener pendiente positiva (cuanto mejor *odd<sup>B</sup>* mejor calidad crediticia tiene el cliente)

 $$ score= {offset + Factor} ~·~ {ln(odds^B)}$$

 Nótese que ahora habrá que establecer de nuevo los puntos de score que doblan el odd ratio (**pdo_0**), y también la puntuación o score considerada como de sobresaliente(**scorecard_points**) y el odd ratio que debería tener ese cliente de *sobresaliente* **odds_0 <sup>*B*</sup>**, con **odds_0 <sup>*B*</sup>** **= 1/odds_0**.

 Así para estimar la puntuación crediticia con `optBinning` hay que establecer tres parámetros para transformar probabilidades de impago a puntuaciones, por ejemplo:   

* **pdo_0** =40
* **scorecard_points** =600
* **odds_0 <sup>*B*</sup>** = 50  (equivalente a **odds_0** =1/50 )




In [ ]:
def scatter_mortality_vs_rt(df, rt_var_pm, cancer, years=[2016, 2022]):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, year in zip(axes, years):
        df_y = df[(df["Year"] == year) &
               (df["Cancer label"].isin(cancer))]

        ax.scatter(
            df_y["ASR (World)"],
            alpha=0.7
        )

        ax.set_title(str(year))
        ax.set_xlabel(f"{rt_var_pm.replace('_per_million', '')}\n(per million inhabitants)")

    axes[0].set_ylabel("Mortality ASR (World)")
    plt.suptitle("Cancer mortality vs radiotherapy availability")
    plt.tight_layout()
    plt.show()

In [ ]:
def scatter_mortality_vs_rt(df, rt_var_pm, cancer, years=[2016, 2022]):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, year in zip(axes, years):
        df_y = df[(df["Year"] == year) &
               (df["Cancer label"].isin(cancer))]

        ax.scatter(
            df_y[rt_var_pm],
            df_y["ASR (World)"],
            alpha=0.7
        )

        ax.set_title(str(year))
        ax.set_xlabel(f"{rt_var_pm.replace('_per_million', '')}\n(per million inhabitants)")

    axes[0].set_ylabel("Mortality ASR (World)")
    plt.suptitle("Cancer mortality vs radiotherapy availability")
    plt.tight_layout()
    plt.show()

# METRICAS DE CARGA DE ENFERMEDAD

In [ ]:
# Calcular primero el total de muertes por país-año-sexo
total_deaths_by_group = (
    df_merged_all.groupby(['Country_harmonized', 'Year', 'Sex'])
    ['Total'].transform('sum')
)

# Proporción de cada tipo de cáncer respecto al total
df_merged_all['cancer_proportion'] = (
    df_merged_all['Total'] / total_deaths_by_group
) * 100


# 3. Métrica de brecha
# Casos estimados que necesitan RT por país

Existen estudios ya publicados donde se muestran cuantos equipos se necesitan po millón de habitantes, estos equipos son linacs esencialmente ya que es, a pesar de

In [ ]:
# Fuente: Barton et al. (2014), Radiotherapy and Oncology, Table 1, p.141
# Proporciones específicas de indicación de RT por tipo de cáncer

rt_indication_rates = {
    'Cervix': 0.71,          # Barton 2014, Table 1
    'Prostate': 0.58,        # Barton 2014, Table 1
    'Breast': 0.87,          # Barton 2014, Table 1
    'Lung': 0.77,            # Barton 2014, Table 1
    #'Colon': 0.04            # Barton 2014, Table 1 (rectum: 0.60)
    "Lip_oc_Pharynx": 0.7
}

df_merged_all['rt_indication_rate'] = (
    df_merged_all['Cancer label'].map(rt_indication_rates)
)

df_merged_all['estimated_rt_need'] = (
    df_merged_all['Total'] * df_merged_all['rt_indication_rate']
)

# 3. TOTAL DE CASOS RT POR PAÍS-AÑO-SEXO
df_merged_all['total_rt_need'] = (
    df_merged_all.groupby(['Country_harmonized', 'Year', 'Sex'])
    ['estimated_rt_need'].transform('sum')
)

# 4. RATIO CASOS POR CENTRO DE RT
df_merged_all['cases_per_rt_center'] = (
    df_merged_all['total_rt_need'] / df_merged_all['RTCenters']
)


In [ ]:
# Déficit de equipos (benchmark OMS: 1 linac por 250,000 hab)
# Crear un dataframe auxiliar con LINACs necesarios por país-año
linacs_needed = (
    df_merged_all
    .groupby(['Country_harmonized', 'Year'], as_index=False)
    .agg({'Population': 'first'})  # Población única por país-año
)

linacs_needed['rt_linac_needed'] = linacs_needed['Population'] / 250000

# To prevent KeyError on re-run, drop existing conflicting columns before merging
columns_to_drop_if_exist = ['rt_linac_needed', 'rt_deficit', 'coverage_index']
for col in columns_to_drop_if_exist:
    if col in df_merged_all.columns:
        df_merged_all = df_merged_all.drop(columns=[col])


# Fusionar de vuelta
df_merged_all = df_merged_all.merge(
    linacs_needed[['Country_harmonized', 'Year', 'rt_linac_needed']],
    on=['Country_harmonized', 'Year'],
    how='left'
)


# Déficit de equipos
df_merged_all['rt_deficit'] = (
    df_merged_all['rt_linac_needed'] - df_merged_all['Linac']
)

# Índice de cobertura (%)
df_merged_all['coverage_index'] = (
    df_merged_all['Linac'] / df_merged_all['rt_linac_needed']
) * 100

print(df_merged_all.head())

In [ ]:

# Comparar proporción de RT por tipo de cáncer
fig, ax = plt.subplots(figsize=(10, 6))

rt_summary = (
    df_merged_all[df_merged_all['Year'] == 2022]
    .groupby('Cancer label', as_index=False)
    .agg({
        'Total': 'sum',
        'estimated_rt_need': 'sum',
        'rt_indication_rate': 'first'
    })
)

ax.bar(rt_summary['Cancer label'], rt_summary['rt_indication_rate'] * 100)
ax.set_ylabel('% de casos que necesitan RT')
ax.set_xlabel('Tipo de cáncer')
ax.set_title('Indicación de Radioterapia por Tipo de Cáncer')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
!pip install pycountry # Install the missing library

def get_iso3(country_name):
    try:
        return pycountry.countries.search_fuzzy(country_name)[0].alpha_3
    except:
        return None

df_merged_all['iso_alpha_3'] = (
    df_merged_all['Country_harmonized'].apply(get_iso3)
)

# Luego usar en el mapa:
fig = px.choropleth(
    df_merged_all[df_merged_all['Year']==2022],
    locations="iso_alpha_3",  # ✅ Usar códigos ISO
    locationmode="ISO-3",
    color="coverage_index",
    hover_data=['Country_harmonized', 'ASR (World)', 'rt_deficit'],
    title="Cobertura de Radioterapia Global 2022"
)

fig.show()

# 2. Scatter con regresión por región
sns.lmplot(
    data=df_merged_all[df_merged_all['Year']==2022],
    x='RTCenters_m', # Corrected column name to RTCenters_m
    y='ASR (World)',
    hue='Sex',
    col='Region Name',
    height=4
)

# 3. Evolución temporal top/bottom países
top_bottom = pd.concat([
    df_merged_all.nlargest(5, 'ASR (World)'),
    df_merged_all.nsmallest(5, 'ASR (World)')
])

sns.lineplot(
    data=top_bottom,
    x='Year',
    y='ASR (World)',
    hue='Country_harmonized',
    style='Sex'
)


In [ ]:
# ✅ Nombre más claro
df_merged_all['total_deaths_all_cancers'] = (
    df_merged_all.groupby(['Country_harmonized', 'Year', 'Sex'])
    ['Total'].transform('sum')
)

# Y luego:
df_merged_all['crude_mortality_rate_per_100k'] = (
    (df_merged_all['total_deaths_all_cancers'] / df_merged_all['Population']) * 100000
)

# La línea de abajo es redundante y errónea, ya que 'total_deaths' no existe y 'crude_mortality_rate_per_100k' ya se calculó.
# df_merged_all['crude_mortality_rate_per_100k'] = (
#     (df_merged_all['total_deaths'] / df_merged_all['Population']) * 100000
# )

In [ ]:
# Promedio de las tasas estandarizadas (más apropiado que suma)
df_merged_all['avg_asr_mortality'] = (
    df_merged_all.groupby(['Country_harmonized', 'Year', 'Sex'])
    ['ASR (World)'].transform('mean')
)

In [ ]:
# Calcular primero el total de muertes por país-año-sexo
total_deaths_by_group = (
    df_merged_all.groupby(['Country_harmonized', 'Year', 'Sex'])
    ['Total'].transform('sum')
)

# Proporción de cada tipo de cáncer respecto al total
df_merged_all['cancer_proportion'] = (
    df_merged_all['Total'] / total_deaths_by_group
) * 100

"Para estimar la demanda de radioterapia por tipo de cáncer, se utilizaron las tasas de utilización óptima (RUR) reportadas por Barton et al. (2014), quienes realizaron una revisión sistemática de guías clínicas basadas en evidencia de organizaciones internacionales (NCCN, NCI, ESMO, Cancer Care Ontario). Las proporciones aplicadas fueron:*

*- Cáncer de mama: 87% (Barton et al., 2014, Table 1, p.141)*  
*- Cáncer de pulmón: 77%*  
*- Cáncer de cuello uterino: 71%*  
*- Cáncer de próstata: 58%*  
*- Cáncer de recto: 60%*  
*- Cáncer de colon: 4%*

*Estas tasas representan la proporción de pacientes con cada tipo de cáncer que tienen indicación de radioterapia de haz externo al menos una vez durante el curso de su enfermedad, basándose en la mejor evidencia disponible sobre eficacia clínica."*

*"El estudio de Barton et al. (2014) reporta que la tasa global de utilización óptima de radioterapia es del 48.3%, lo que implica que aproximadamente la mitad de todos los pacientes con cáncer deberían recibir radioterapia en algún momento de su tratamiento. Sin embargo, estudios previos han demostrado que las tasas de utilización reales son significativamente inferiores a las óptimas en países desarrollados como Australia, Reino Unido, Canadá y España (Barton & Delaney, 2011; Erridge et al., 2007), lo que sugiere una brecha persistente entre la demanda óptima y el acceso real a radioterapia."*

## Re-run plotting loop

### Subtask:
Execute cell `sOqz_3Wjvois` again to run the plotting loop with the corrected function.


In [ ]:
summary_2022 = (
    df_merged_all[df_merged_all['Year'] == 2022]
    .groupby(['Country_harmonized', 'Sex'], as_index=False)
    .agg({
        'Total': 'sum',
        'Crude rate': 'mean',
        'ASR (World)': 'mean',
        'Population': 'first',
        'coverage_index': 'first',  # Ya calculado antes
        'rt_deficit': 'first'
    })
    .rename(columns={
        'Total': 'total_cancer_deaths',
        'Crude rate': 'crude_mortality_per_100k',
        'ASR (World)': 'avg_asr_all_cancers'
    })
    .sort_values('crude_mortality_per_100k', ascending=False)
)

print(summary_2022.head(10))

# Fase 4, modelización, modelo de regresión lineal simple.

variable dependiente:
Mortalidad por cáncer en ASR world

Variables dependientes: Linac_m RTCenters_m

Estratificación: por sexo, por tipo de cancer y por año

In [ ]:


# Ejemplo: Modelo para cáncer de pulmón en hombres (2022)
df_model = df_merged_all[
    (df_merged_all['Cancer label'] == 'Lung') &
    (df_merged_all['Sex'] == 'Male') &
    (df_merged_all['Year'] == 2022)
].copy()

# Variable dependiente
y = df_model['ASR (World)']

# Variable independiente
X = df_model[['Linac_m']]
X = sm.add_constant(X)  # Añadir intercepto

# Ajustar modelo
model = sm.OLS(y, X).fit()

# Ver resultados
print(model.summary())

#modelo multivariable

Se tomara en cuenta el resto de las estructuras, así como la población y densidad, PIB per capita y casto en salud per capita

Variables de carga de enfermedad:

cancer_proportion (% del cáncer respecto al total)
estimated_rt_need (casos que necesitan RT)
coverage_index (cobertura de RT)

In [ ]:
# Modelo multivariable
X = df_model[[
    'Linac_m',
    'RTCenters_m',
    'coverage_index',
    'cancer_proportion'
]]

X = sm.add_constant(X)
y = df_model['ASR (World)']

model_multi = sm.OLS(y, X).fit()
print(model_multi.summary())

4.3. Modelo de Puntuación de Riesgo (Score)Objetivo: Crear un índice de riesgo que combine:Mortalidad observada (ASR World)Déficit de infraestructura (rt_deficit)Cobertura de RT (coverage_index)Carga de enfermedad (total_deaths)

In [ ]:
# Normalizar variables entre 0-100


scaler = MinMaxScaler(feature_range=(0, 100))

df_score = df_merged_all.copy()

# Normalizar (mayor mortalidad = peor score)
df_score['mortality_score'] = 100 - scaler.fit_transform(
    df_score[['ASR (World)']]
)

# Normalizar (menor cobertura = peor score)
df_score['coverage_score'] = scaler.fit_transform(
    df_score[['coverage_index']]
)

# Score final (ejemplo simple)
df_score['risk_score'] = (
    0.5 * df_score['mortality_score'] +
    0.3 * df_score['coverage_score'] +
    0.2 * (100 - df_score['cancer_proportion'])  # Menor proporción = mejor
)

# Clasificar en categorías de riesgo
df_score['risk_category'] = pd.cut(
    df_score['risk_score'],
    bins=[0, 25, 50, 75, 100],
    labels=['Bajo', 'Medio', 'Alto', 'Muy Alto']
)

FASE 5: VALIDACIÓN Y VISUALIZACIÓN 🔜 (PENDIENTE)Lo que deberás hacer:Validación de modelos:

R² ajustado
Test de significancia de coeficientes
Análisis de residuos
VIF (multicolinealidad)

Visualizaciones finales:

Mapas de calor de riesgo por país
Scatter plots con líneas de regresión
Gráficos de barra con scores por país
Comparación 2016 vs 2022

Interpretación:

¿Qué países tienen peor situación?
¿La infraestructura RT explica la mortalidad?
¿Hay diferencias por sexo/tipo de cáncer?

In [ ]:
# Resumen por país (2022)
summary_rt_demand = (
    df_merged_all[df_merged_all['Year'] == 2022]
    .groupby('Country_harmonized', as_index=False)
    .agg({
        'Total': 'sum',  # Total muertes por cáncer
        'estimated_rt_need': 'sum',  # Total casos que necesitan RT
        'Linac': 'first',
        'rt_linac_needed': 'first',
        'rt_deficit': 'first',
        'coverage_index': 'first',
        'Population': 'first'
    })
    .sort_values('rt_deficit', ascending=False)
)

print("Top 10 países con mayor déficit de LINACs:")
print(summary_rt_demand.head(10))

# Gráfico


fig, ax = plt.subplots(figsize=(12, 6))

top10_deficit = summary_rt_demand.nlargest(10, 'rt_deficit')

ax.barh(
    top10_deficit['Country_harmonized'],
    top10_deficit['rt_deficit'],
    color='#C44E52'
)

ax.set_xlabel('Déficit de LINACs')
ax.set_title('Top 10 Países con Mayor Déficit de Aceleradores Lineales (2022)')
plt.tight_layout()
plt.savefig(f"{fig_dir}/deficit_linacs_top10.jpg", dpi=300)
plt.show()

FASE 2: Análisis Univariante y BivarianteObjetivo: Identificar qué variables están asociadas con la mortalidad.2.1. Variables Continuas vs Mortalidad Code import scipy.stats as stats



In [ ]:


# Lista de variables de infraestructura RT
rt_vars_continuous = [
    'Linac_m',
    'RTCenters_m',
    'Brachytherapy_m',
    'coverage_index',
    'rt_deficit'
]

# Test de correlación de Pearson
print("="*60)
print("CORRELACIÓN CON MORTALIDAD (ASR World)")
print("="*60)

for var in rt_vars_continuous:
    # Eliminar NaN
    df_clean = df_merged_all[[var, 'ASR (World)']].dropna()

    # Correlación de Pearson
    r, p_value = stats.pearsonr(
        df_clean[var],
        df_clean['ASR (World)']
    )

    print(f"\n{var:30s} | r={r:6.3f} | p-valor={p_value:.4f}")

    # Interpretación
    if p_value < 0.05:
        if r > 0:
            print(f"  ✅ Correlación POSITIVA significativa")
        else:
            print(f"  ✅ Correlación NEGATIVA significativa")
    else:
        print(f"  ❌ NO significativa")

2.2. Gráficos de Dispersión con Regresión Code


In [ ]:

# Filtrar datos 2022
df_2022 = df_merged_all[df_merged_all['Year'] == 2022].copy()

# Crear grid de gráficos
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, var in enumerate(rt_vars_continuous):
    sns.regplot(
        data=df_2022,
        x=var,
        y='ASR (World)',
        ax=axes[i],
        scatter_kws={'alpha': 0.5},
        line_kws={'color': 'red'}
    )

    axes[i].set_title(f'Mortalidad vs {var}')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Mortalidad ASR (World)')

plt.tight_layout()
plt.savefig(f"{fig_dir}/univariate_rt_vs_mortality.jpg", dpi=300)
plt.show()





FASE 3: Binning/Tramificación (ADAPTADO)Objetivo: Agrupar países en categorías de infraestructura RT.3.1. Tramificación Óptima de Variables Continuas Code

In [ ]:
# Preparar datos para 2022, estratificado por sexo
df_model = df_merged_all[
    (df_merged_all['Year'] == 2022) &
    (df_merged_all['Sex'] == 'Male') &
    (df_merged_all['Cancer label'] == 'Lung')
].copy()

# Eliminar NaN
df_model = df_model.dropna(subset=['ASR (World)', 'Linac_m'])

# Variable continua: Mortalidad (necesitamos convertirla a binaria)
# Definir "alta mortalidad" como por encima de la mediana
median_mortality = df_model['ASR (World)'].median()
df_model['high_mortality'] = (
    df_model['ASR (World)'] > median_mortality
).astype(int)

# Tramificación de Linac_m
variable = "Linac_m"
X = df_model[variable].values
Y = df_model['high_mortality'].values

optb = OptimalBinning(name=variable, dtype="numerical", solver="cp")
optb.fit(X, Y)

print("Cortes óptimos para Linac_m:")
print(optb.splits)

# Tabla de binning
binning_table = optb.binning_table
binning_table.build()

Salida esperada: Code Cortes óptimos para Linac_m:
[1.5  3.0  5.0  8.0]

   Bin         Count  Count (%)  Non-event  Event  Event rate    WoE       IV
0  (-inf, 1.5)    15    0.25        3        12     0.80      1.386    0.350
1  [1.5, 3.0)     20    0.33        8        12     0.60      0.405    0.055
2  [3.0, 5.0)     15    0.25        9         6     0.40     -0.405    0.045
3  [5.0, 8.0)      8    0.13        6         2     0.25     -1.099    0.180
4  [8.0, inf)      2    0.03        2         0     0.00     -2.773    0.370Interpretación:Países con menos de 1.5 LINACs/millón tienen 80% de probabilidad de alta mortalidadPaíses con más de 8 LINACs/millón tienen 0% de probabilidad de alta mortalidad


3.2. Visualización de Binning Code binning_table.plot(metric="event_rate")


In [ ]:
plt.title('Tasa de Alta Mortalidad por Nivel de LINACs/millón')
plt.xlabel('LINACs por millón de habitantes')
plt.ylabel('Probabilidad de Alta Mortalidad')
plt.savefig(f"{fig_dir}/binning_linacs_mortality.jpg", dpi=300)
plt.show()



FASE 4: Transformación WOE (Weight of Evidence)Objetivo: Convertir variables categorizadas en valores continuos que reflejen su asociación con la mortalidad. Code

In [ ]:
# Transformación WOE
df_model['Linac_m_woe'] = optb.transform(X, metric="woe")

# Ver distribución
print(df_model['Linac_m_woe'].value_counts().sort_index())

# Gráfico: Relación linealizada
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(
    df_model['Linac_m_woe'],
    df_model['ASR (World)'],
    alpha=0.6
)

# Línea de regresión
sns.regplot(
    data=df_model,
    x='Linac_m_woe',
    y='ASR (World)',
    scatter=False,
    color='red',
    ax=ax
)

ax.set_xlabel('WOE de LINACs/millón')
ax.set_ylabel('Mortalidad ASR (World)')
ax.set_title('Relación Linealizada: WOE vs Mortalidad')
plt.savefig(f"{fig_dir}/woe_linacs_vs_mortality.jpg", dpi=300)
plt.show()

FASE 5: Modelo de Regresión LogísticaObjetivo: Predecir probabilidad de "alta mortalidad" basándose en infraestructura RT. Code

In [ ]:
# Preparar variables transformadas WOE
rt_vars_woe = ['Linac_m_woe', 'RTCenters_m_woe', 'coverage_index_woe']

# (Asumiendo que ya hiciste binning de todas las variables)
# Para simplificar, usamos solo Linac_m_woe

X_model = df_model[['Linac_m_woe']].values
y_model = df_model['high_mortality'].values

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.25, random_state=42, stratify=y_model
)

# Ajustar modelo
logreg = LogisticRegression()
logreg.fit(X_train, y_train)

# Predicciones
y_pred_proba = logreg.predict_proba(X_test)[:, 1]
y_pred = logreg.predict(X_test)

# Evaluación
print("="*60)
print("RESULTADOS DEL MODELO")
print("="*60)
print("\nCoeficientes:")
print(f"  Intercepto: {logreg.intercept_[0]:.4f}")
print(f"  Linac_m_woe: {logreg.coef_[0][0]:.4f}")

print("\nMétricas:")
print(f"  AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.3f}")

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

FASE 6: Scorecard (Tarjeta de Puntuación)Objetivo: Convertir probabilidades en un Índice de Riesgo Poblacional (0-1000 puntos). Code


# Tabla de puntuación
scorecard.table()Salida esperada: Code Variable    Bin          Points
Linac_m     (-inf, 1.5)   -80
Linac_m     [1.5, 3.0)    -30
Linac_m     [3.0, 5.0)     20
Linac_m     [5.0, 8.0)     60
Linac_m     [8.0, inf)    100Interpretación:España con 3.5 LINACs/millón → +20 puntosIndia con 0.8 LINACs/millón → -80 puntosFASE 7: Aplicar Score a Todos los Países Code

# Calcular score para cada país
df_2022['risk_score'] = scorecard.score(df_2022[['Linac_m']])

# Normalizar a 0-100
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 100))
df_2022['risk_score_normalized'] = scaler.fit_transform(
    df_2022[['risk_score']]
)

# Categorizar
df_2022['risk_category'] = pd.cut(
    df_2022['risk_score_normalized'],
    bins=[0, 25, 50, 75, 100],
    labels=['Muy Alto', 'Alto', 'Medio', 'Bajo']
)

# Top 10 países con peor score
worst_10 = df_2022.nsmallest(10, 'risk_score_normalized')

print("TOP 10 PAÍSES CON MAYOR RIESGO POBLACIONAL:")
print(worst_10[['Country_harmonized', 'Linac_m', 'ASR (World)',
                'risk_score_normalized', 'risk_category']])

📊 VENTAJAS DE ESTE ENFOQUE✅ Metodología Rigurosa: Usa técnicas probadas en credit scoring✅ Interpretabilidad: Los WOE y scores son fáciles de explicar✅ Manejo de No-Linealidad: El binning captura relaciones complejas✅ Robustez: Menos sensible a outliers que regresión lineal simple✅ Output Accionable: Un score de 0-100 es fácil de comunicar a policy makers🎯 ESTRUCTURA FINAL DEL TFMCapítulo 3: Metodología3.1. Análisis Univariante y Bivariante  
3.2. Tramificación Óptima de Variables (Optimal Binning)  
3.3. Transformación WOE (Weight of Evidence)  
3.4. Modelo de Regresión Logística  
3.5. Construcción del Índice de Riesgo Poblacional (Scorecard)  Capítulo 4: Resultados4.1. Variables Asociadas con Mortalidad  
4.2. Tabla de Puntuación por País  
4.3. Mapa de Riesgo Poblacional en Europa  
4.4. Anál